# Erlang C Calculator — Full Integration Tests

This notebook tests the connected calculation workflow using the real 2021–2024 CDR files. Place it inside `tests`, keep the CSV files in `tests/data`, select `Erlang C (.venv)`, and run the cells in order. The full forecast cell can take several minutes.

## Setup 

In [ ]:
import sys
import time
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
PROJECT_DIR = cwd if (cwd / "calculator.py").exists() else cwd.parent
if not (PROJECT_DIR / "calculator.py").exists():
    raise FileNotFoundError("Could not find calculator.py. Place this notebook inside tests.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from calculator import (
    build_stl_forecast, build_dashboard_aggregates, build_shift_requirements,
    calculate_schedule_headcount, build_monthly_agent_schedule,
)

DATA_CANDIDATES = [cwd / "data", cwd / "tests" / "data"]
DATA_DIR = next((folder for folder in DATA_CANDIDATES if folder.exists()), DATA_CANDIDATES[0])
CDR_FILES = [DATA_DIR / f"cdr_{year}.csv" for year in range(2021, 2025)]
INTEGRATION_RESULTS = []
STATE = {}

def record(test_id, description, passed, expected, actual, details=""):
    INTEGRATION_RESULTS[:] = [row for row in INTEGRATION_RESULTS if row["test"] != test_id]
    status = "PASS" if passed else "FAIL"
    INTEGRATION_RESULTS.append({"test": test_id, "description": description, "status": status, "expected": str(expected), "actual": str(actual), "details": details})
    print(f"{status}: {test_id} — {description}")
    print("Expected:", expected)
    print("Actual:  ", actual)
    if details: print("Details: ", details)
    return passed

print("Setup complete")
print("Python:", sys.executable)
print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)

## Integration Test 1 — Find all four real datasets

In [ ]:
missing = [path.name for path in CDR_FILES if not path.exists()]
for path in CDR_FILES:
    print(path.name, "FOUND" if path.exists() else "MISSING")
record("IT-01", "All real CDR files are available", not missing, "Four CSV files found", f"{4-len(missing)} found; missing={missing}")

## Integration Test 2 — Generate the complete 2025 forecast

This connects CSV reading, cleaning, interval creation, STL forecasting, AHT estimation, Erlang C staffing and summary generation. It may take several minutes.

In [ ]:
if any(not path.exists() for path in CDR_FILES):
    record("IT-02", "Full four-year forecast completes", False, "Forecast created", "Dataset file missing")
else:
    started = time.perf_counter()
    try:
        forecast, forecast_summary = build_stl_forecast(
            file_paths=CDR_FILES, filenames=[path.name for path in CDR_FILES],
            interval_minutes=30, forecast_days=365, trend_lookback_days=90,
            target_seconds=20, target_service_level=80, shrinkage=30, max_agents=10000,
        )
        elapsed = time.perf_counter() - started
        STATE["forecast"] = forecast
        STATE["forecast_summary"] = forecast_summary
        STATE["forecast_seconds"] = elapsed
        passed = forecast_summary.get("historical_years") == [2021, 2022, 2023, 2024] and forecast_summary.get("output_year") == 2025
        record("IT-02", "Full four-year forecast completes", passed, "Years 2021–2024 produce 2025", {"historical_years": forecast_summary.get("historical_years"), "output_year": forecast_summary.get("output_year")}, f"Elapsed: {elapsed:.2f} seconds")
    except Exception as error:
        record("IT-02", "Full four-year forecast completes", False, "Forecast created", f"{type(error).__name__}: {error}")

## Integration Test 3 — Validate forecast structure and values

In [ ]:
if "forecast" not in STATE:
    record("IT-03", "Forecast output is structurally valid", False, "IT-02 completed", "Forecast unavailable")
else:
    forecast = STATE["forecast"]
    required_columns = {"interval_start", "call_volume", "aht_seconds", "traffic_erlangs", "raw_agents", "scheduled_agents", "day_of_year"}
    checks = {
        "17520_rows": len(forecast) == 365 * 48,
        "required_columns": required_columns.issubset(forecast.columns),
        "no_missing_values": not forecast.isna().any().any(),
        "calls_nonnegative": bool((forecast["call_volume"] >= 0).all()),
        "agents_nonnegative": bool((forecast["scheduled_agents"] >= 0).all()),
        "day_of_year_1_to_365": forecast["day_of_year"].min() == 1 and forecast["day_of_year"].max() == 365,
    }
    record("IT-03", "Forecast output is structurally valid", all(checks.values()), "All checks True", checks)
    display(forecast.head())

## Integration Test 4 — Create all dashboard aggregates

In [ ]:
if "forecast" not in STATE:
    record("IT-04", "Dashboard aggregates are created", False, "IT-02 completed", "Forecast unavailable")
else:
    try:
        charts = build_dashboard_aggregates(STATE["forecast"])
        STATE["charts"] = charts
        expected_keys = {"monthly", "daily", "weekday", "time_of_day"}
        checks = {"all_sections": expected_keys.issubset(charts), "monthly_not_empty": bool(charts.get("monthly")), "daily_not_empty": bool(charts.get("daily")), "weekday_not_empty": bool(charts.get("weekday")), "time_not_empty": bool(charts.get("time_of_day"))}
        record("IT-04", "Dashboard aggregates are created", all(checks.values()), "Four non-empty chart sections", checks)
    except Exception as error:
        record("IT-04", "Dashboard aggregates are created", False, "Aggregates created", f"{type(error).__name__}: {error}")

## Integration Test 5 — Build January shift requirements

In [ ]:
if "forecast" not in STATE:
    record("IT-05", "January shift requirements are created", False, "IT-02 completed", "Forecast unavailable")
else:
    try:
        shift_requirements = build_shift_requirements(STATE["forecast"], year=2025, month=1)
        STATE["shift_requirements"] = shift_requirements
        checks = {"93_rows": len(shift_requirements) == 31 * 3, "three_shifts": set(shift_requirements["shift_code"]) == {"NIGHT", "MORNING", "EVENING"}, "nonnegative": bool((shift_requirements["required_agents"] >= 0).all())}
        record("IT-05", "January shift requirements are created", all(checks.values()), "31 days × 3 shifts = 93 rows", checks)
        display(shift_requirements.head(6))
    except Exception as error:
        record("IT-05", "January shift requirements are created", False, "93 shift rows", f"{type(error).__name__}: {error}")

## Integration Test 6 — Calculate headcount from real forecast requirements

In [ ]:
if "shift_requirements" not in STATE:
    record("IT-06", "Required headcount is calculated", False, "IT-05 completed", "Requirements unavailable")
else:
    try:
        headcount = calculate_schedule_headcount(STATE["shift_requirements"], working_days_per_week=5)
        STATE["headcount"] = headcount
        record("IT-06", "Required headcount is calculated", isinstance(headcount, int) and headcount > 0, "Positive integer", headcount)
    except Exception as error:
        record("IT-06", "Required headcount is calculated", False, "Positive integer", f"{type(error).__name__}: {error}")

## Integration Test 7 — Generate the January monthly schedule

In [ ]:
if "forecast" not in STATE:
    record("IT-07", "January schedule is generated", False, "IT-02 completed", "Forecast unavailable")
else:
    try:
        schedule, schedule_summary = build_monthly_agent_schedule(STATE["forecast"], year=2025, month=1, agent_count=None)
        STATE["schedule"] = schedule
        STATE["schedule_summary"] = schedule_summary
        checks = {"not_empty": not schedule.empty, "coverage_ok": schedule_summary.get("coverage_ok") is True, "no_shortage": schedule_summary.get("coverage_shortage") == 0}
        record("IT-07", "January schedule is generated", all(checks.values()), "Non-empty schedule with full coverage", checks, f"Agents: {schedule_summary.get('agent_count')}")
        display(schedule.head())
    except Exception as error:
        record("IT-07", "January schedule is generated", False, "Schedule generated", f"{type(error).__name__}: {error}")

## Integration Test 8 — Validate roster uniqueness and five-day weekly limit

In [ ]:
if "schedule" not in STATE:
    record("IT-08", "Roster rules are satisfied", False, "IT-07 completed", "Schedule unavailable")
else:
    schedule = STATE["schedule"].copy()
    schedule["date_dt"] = pd.to_datetime(schedule["date"])
    schedule["week_start"] = schedule["date_dt"] - pd.to_timedelta(schedule["date_dt"].dt.weekday, unit="D")
    duplicate_agent_dates = int(schedule.duplicated(["agent_id", "date"]).sum())
    weekly_counts = schedule.loc[schedule["status"] == "WORK"].groupby(["agent_id", "week_start"]).size()
    maximum_weekly_days = int(weekly_counts.max()) if not weekly_counts.empty else 0
    checks = {"one_row_per_agent_date": duplicate_agent_dates == 0, "maximum_five_workdays": maximum_weekly_days <= 5}
    record("IT-08", "Roster rules are satisfied", all(checks.values()), "No duplicate agent-date; maximum 5 workdays/week", {**checks, "maximum_weekly_days": maximum_weekly_days})

## Integration Test 9 — Check the initial schedule's 8-hour rest rule

This may reveal a real scheduling defect. A failure must be recorded; do not change the test merely to make it pass.

In [ ]:
if "schedule" not in STATE:
    record("IT-09", "Every consecutive shift has at least 8 hours rest", False, "IT-07 completed", "Schedule unavailable")
else:
    shift_hours = {"NIGHT": (0, 8), "MORNING": (8, 16), "EVENING": (16, 24)}
    work = STATE["schedule"].loc[STATE["schedule"]["status"] == "WORK"].copy()
    periods = []
    for row in work.itertuples(index=False):
        start_hour, end_hour = shift_hours[row.shift_code]
        day = pd.Timestamp(row.date)
        start = day + pd.Timedelta(hours=start_hour)
        end = day + pd.Timedelta(hours=end_hour)
        periods.append({"agent_id": row.agent_id, "date": row.date, "shift_code": row.shift_code, "start": start, "end": end})
    periods = pd.DataFrame(periods).sort_values(["agent_id", "start"])
    violations = []
    for agent_id, agent_rows in periods.groupby("agent_id"):
        records = list(agent_rows.to_dict(orient="records"))
        for previous, current in zip(records, records[1:]):
            rest_hours = (current["start"] - previous["end"]).total_seconds() / 3600
            if rest_hours < 8:
                violations.append({"agent_id": agent_id, "previous_date": previous["date"], "previous_shift": previous["shift_code"], "next_date": current["date"], "next_shift": current["shift_code"], "rest_hours": rest_hours})
    STATE["rest_violations"] = violations
    record("IT-09", "Every consecutive shift has at least 8 hours rest", len(violations) == 0, "0 violations", len(violations), "First violations shown below if present")
    if violations: display(pd.DataFrame(violations).head(10))

## Integration Test 12 — Verify important forecast summary values

In [ ]:
if "forecast_summary" not in STATE:
    record("IT-12", "Forecast summary values are sensible", False, "IT-02 completed", "Summary unavailable")
else:
    summary = STATE["forecast_summary"]
    checks = {
        "dataset_count_4": summary.get("dataset_count") == 4,
        "days_365": summary.get("days") == 365,
        "interval_30": summary.get("interval_minutes") == 30,
        "predicted_calls_nonnegative": summary.get("total_predicted_calls", -1) >= 0,
        "scheduled_agents_nonnegative": summary.get("maximum_scheduled_agents", -1) >= 0,
    }
    record("IT-12", "Forecast summary values are sensible", all(checks.values()), "All checks True", checks)
    display(pd.DataFrame([{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}]).T.rename(columns={0: "value"}))

## Final integration-test report — run after Tests 1–12

In [ ]:
expected_ids = [f"IT-{number:02d}" for number in range(1, 13)]
completed = {row["test"] for row in INTEGRATION_RESULTS}
not_run = [test_id for test_id in expected_ids if test_id not in completed]
passed = sum(row["status"] == "PASS" for row in INTEGRATION_RESULTS)
failed = sum(row["status"] == "FAIL" for row in INTEGRATION_RESULTS)
table_rows = []
for test_id in expected_ids:
    row = next((item for item in INTEGRATION_RESULTS if item["test"] == test_id), None)
    if row:
        table_rows.append(f"| {row['test']} | {row['description']} | {row['status']} |")
    else:
        table_rows.append(f"| {test_id} | Not run | NOT RUN |")
final_status = "PASS" if failed == 0 and not not_run else ("FAIL" if failed else "INCOMPLETE")
rest_note = "No rest violations detected." if not STATE.get("rest_violations") else f"{len(STATE['rest_violations'])} initial-roster rest violation(s) detected; review IT-09."
report = f"""
# Erlang C Integration Testing Report

## Test information

- Test date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- Historical datasets: 2021, 2022, 2023 and 2024
- Forecast year: 2025
- Forecast interval: 30 minutes
- Forecast duration: 365 days
- Forecast execution time: {STATE.get('forecast_seconds', 'Not available')} seconds

## Overall results

| Result | Count |
|---|---:|
| Expected tests | {len(expected_ids)} |
| Passed | {passed} |
| Failed | {failed} |
| Not run | {len(not_run)} |

## Detailed results

| Test | Description | Result |
|---|---|---|
{chr(10).join(table_rows)}

## Scheduling safety note

{rest_note}

## Final status

{final_status}

{('All integration tests passed.' if final_status == 'PASS' else ('One or more integration tests found an issue that should be reviewed.' if final_status == 'FAIL' else 'Run all cells in order before finalizing.'))}
"""
display(Markdown(report))